Install the Gemini library

In [2]:
!pip -q install -U google-genai

Connect your API key

In [3]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

Gemini API connected successfully!


first LLM API call (gemini flash 3.8 wa always in high demand)

In [5]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain what Artificial Intelligence is in 3 simple sentences."
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [8]:
import time
import random
from google.genai.errors import ServerError

response = None

for attempt in range(5):
    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="Explain Artificial Intelligence in 3 simple sentences."
        )
        print(response.text)
        break

    except ServerError as e:
        if "503" not in str(e):
            raise

        wait = (2 ** attempt) + random.uniform(0, 2)
        print(f"Attempt {attempt + 1} failed. Retrying in {wait:.1f} seconds...")
        time.sleep(wait)

if response is None:
    print("Gemini is still unavailable. We will try another model.")

Attempt 1 failed. Retrying in 2.4 seconds...
Attempt 2 failed. Retrying in 3.2 seconds...
Attempt 3 failed. Retrying in 5.5 seconds...
Attempt 4 failed. Retrying in 9.8 seconds...
Attempt 5 failed. Retrying in 16.3 seconds...
Gemini is still unavailable. We will try another model.


In [9]:
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026


Trying another model

In [10]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain what Artificial Intelligence is in 3 simple sentences."
)

print(response.text)

Artificial Intelligence (AI) is a branch of computer science that creates machines capable of mimicking human thinking and problem-solving. These systems learn from vast amounts of data to recognize patterns, make decisions, and improve themselves over time. Common examples include virtual assistants, recommendation algorithms, and self-driving cars.


Token counting

In [11]:
MODEL = "gemini-3.5-flash-lite"

In [12]:
token_info = client.models.count_tokens(
    model=MODEL,
    contents="Explain what Artificial Intelligence is in 3 simple sentences."
)

print("Input tokens:", token_info.total_tokens)

response = client.models.generate_content(
    model=MODEL,
    contents="Explain what Artificial Intelligence is in 3 simple sentences."
)

output_tokens = response.usage_metadata.candidates_token_count

print("Output tokens:", output_tokens)

Input tokens: 12
Output tokens: 63


Estimating the API cost

For Gemini 3.5 Flash Lite, Google's published standard paid-tier rates are:
ai.google.dev

- Input: $ 0.30 per 1 million tokens

- Output: $ 2.50 per 1 million tokens

In [13]:
input_tokens = 12
output_tokens = 63

input_cost = (input_tokens / 1_000_000) * 0.30
output_cost = (output_tokens / 1_000_000) * 2.50

total_cost = input_cost + output_cost

print("Input cost: $", input_cost)
print("Output cost: $", output_cost)
print("Total estimated cost: $", total_cost)
print("Actual free-tier cost: $0")

Input cost: $ 3.6e-06
Output cost: $ 0.0001575
Total estimated cost: $ 0.0001611
Actual free-tier cost: $0


Item	Result:

Model	Gemini: 3.5 Flash Lite

Input tokens:	12

Output tokens:	63

Estimated paid cost: Calculated above

Actual free-tier cost	$0

Adding conversation memory

In [14]:
chat_history = []

def chat(user_message):
    chat_history.append({
        "role": "user",
        "parts": [{"text": user_message}]
    })

    response = client.models.generate_content(
        model=MODEL,
        contents=chat_history
    )

    chat_history.append({
        "role": "model",
        "parts": [{"text": response.text}]
    })

    return response.text

In [15]:
print(chat("My name is Alex."))
print(chat("What is my name?"))

Nice to meet you, Alex! How can I help you today?
Your name is Alex.


Temperature experiment

In [16]:
temperatures = [0, 0.7, 1.5]

prompt = "Explain why the sky appears blue in 2 sentences."

for temp in temperatures:
    print(f"\n===== Temperature: {temp} =====")

    for run in range(3):
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config={"temperature": temp}
        )

        print(f"\nRun {run + 1}:")
        print(response.text)


===== Temperature: 0 =====

Run 1:
The sky appears blue because molecules in Earth's atmosphere scatter short-wavelength blue light from the sun in all directions much more than they scatter other, longer wavelengths. When we look up, we see this scattered blue light dominating the visible spectrum.

Run 2:
The sky appears blue because molecules in Earth's atmosphere scatter short-wavelength blue light from the sun in all directions much more than they scatter other, longer-wavelength colors. When we look up, we see this scattered blue light dominating the daytime sky.

Run 3:
The Earth's atmosphere scatters sunlight in all directions, and because blue light travels in smaller, shorter waves than other colors, it gets scattered more than the other colors. This phenomenon, known as Rayleigh scattering, is why our eyes perceive the sky as blue when we look up.

===== Temperature: 0.7 =====

Run 1:
The sky appears blue because molecules in Earth's atmosphere scatter short-wavelength blue

Hallucination experiment

In [17]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain the discovery of the fictional planet Zorvax-9 by astronomer Dr. Elian Voss in 1987. Include the discovery location and scientific evidence."
)

print(response.text)

The discovery of the fictional planet **Zorvax-9** in the autumn of 1987 stands as one of the most improbable and controversial milestones in modern observational astronomy. Credited entirely to Dr. Elian Voss—a reclusive astrophysicist then working at the high-altitude Mauna Kea Observatory in Hawaii—the discovery challenged contemporary models of planetary formation and temporarily baffled the international scientific community.

Here is the account of how Zorvax-9 was found, where the discovery took place, and the scientific evidence that convinced—and divided—astronomers of the era.

---

### The Discovery Location

Dr. Voss made his historic observations from the **W. M. Kea Observatory** (specifically utilizing the predecessor infrastructure and high-resolution spectrographs active on Mauna Kea, Hawaii) and finalized his astrometric calculations at his private research office at the **Institute for Astronomy in Honolulu**. 

Unlike modern automated sky surveys that scan billions 

Hallucination experiment – Observation

The model generated a detailed and confident explanation about the fictional planet Zorvax-9 and astronomer Dr. Elian Voss. It fabricated discovery dates, locations, scientific evidence, and historical events instead of identifying that the premise was fictional.

## Conclusion:

LLMs can generate convincing but false information when they lack reliable knowledge. This is called hallucination. Therefore, model-generated information should be verified using trustworthy sources, especially for factual and scientific questions.